# DAY 05 - TASK 02: DATA AUGMENTATION VÀ HỌC CHUYỂN GIAO VỚI MOBILENETV3 TRÊN CIFAR-10

### Mục tiêu thực hành:
1. **Khảo sát vai trò của Data Augmentation**: Đánh giá ảnh hưởng của các phép tăng cường dữ liệu hợp ngữ nghĩa (RandomCrop, RandomHorizontalFlip) trên mạng CNN tự huấn luyện từ đầu (train from scratch).
2. **So sánh mô hình tự học với Pretrained MobileNetV3-Small**: Đo lường bước nhảy vọt về chất lượng khi tận dụng biểu diễn đặc trưng ImageNet trên dữ liệu nhỏ (2,000 ảnh).
3. **Đánh giá đa chiều và trực quan hóa lỗi**: Phân tích Accuracy, Macro-F1, ma trận nhầm lẫn và hiển thị trực tiếp 5 mẫu ảnh đoán đúng / 5 mẫu ảnh đoán sai của từng cấu hình.



## Bước 1: Thiết lập môi trường và cấu hình Thí nghiệm
Import các module từ thư viện lõi `srcs.libraries` và pipeline `srcs.pipelines.day05_task02_transfer_learning`.



In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.libraries.seeding import seed_everything
from srcs.libraries.training import device
from srcs.libraries.models import count_parameters
from srcs.pipelines.day05_task02_transfer_learning import (
    SmallCNN,
    build_mobilenet_v3_small,
    prepare_partition_indices,
    create_dataloaders,
    train_single_configuration,
    render_comparative_curves,
    render_confusion_matrices,
    render_example_predictions,
    CLASSES,
    DEFAULT_LR
)

seed_everything(42)
comp_device = torch.device(device())
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
print(f"PyTorch Version: {torch.__version__} | Device: {comp_device}")
print(f"Thư mục lưu trữ dữ liệu (Gốc dự án): {DATA_DIR}")



## Bước 2: Định nghĩa 3 cấu hình Thí nghiệm Đối chứng
3 cấu hình được so sánh dưới cùng điều kiện kiểm thử:
1. `cnn_baseline`: SmallCNN 2 blocks (268,650 tham số), ảnh $32 \times 32$, không áp dụng Augmentation.
2. `cnn_augmentation`: SmallCNN 2 blocks (268,650 tham số), ảnh $32 \times 32$, có áp dụng Augmentation trên tập train.
3. `mobilenetv3_small_pretrained`: MobileNetV3-Small pretrained đóng băng backbone (chỉ huấn luyện 10,250 tham số tại head), ảnh $224 \times 224$, có áp dụng Augmentation trên tập train.



In [ ]:
demo_cnn = SmallCNN(len(CLASSES))
demo_mobilenet = build_mobilenet_v3_small(len(CLASSES))

print(f"Số tham số huấn luyện SmallCNN:               {count_parameters(demo_cnn):,}")
print(f"Số tham số huấn luyện MobileNetV3 Pretrained: {count_parameters(demo_mobilenet):,}")



## Bước 3: Chuẩn bị dữ liệu CIFAR-10 và Phân chia cố định (Fixed Split)
Tách tập dữ liệu thành 2,000 mẫu train và 500 mẫu validation theo seed 42.
Tập validation luôn được giữ cố định không tăng cường dữ liệu.



In [ ]:
raw_dataset, train_idx, val_idx = prepare_partition_indices(
    data_dir=DATA_DIR,
    train_size=2000,
    val_size=500,
    seed=42
)
print(f"Số lượng mẫu huấn luyện: {len(train_idx):,} | Số lượng mẫu kiểm định: {len(val_idx):,}")
print(f"Danh sách các phân lớp: {CLASSES}")



## Bước 4: Thực thi Huấn luyện 3 Cấu hình Thí nghiệm qua 4 Epochs


In [ ]:
configs = [
    {"name": "cnn_baseline", "augment": False, "pretrained": False, "model_fn": lambda: SmallCNN(len(CLASSES)), "frozen": False},
    {"name": "cnn_augmentation", "augment": True, "pretrained": False, "model_fn": lambda: SmallCNN(len(CLASSES)), "frozen": False},
    {"name": "mobilenetv3_small_pretrained", "augment": True, "pretrained": True, "model_fn": lambda: build_mobilenet_v3_small(len(CLASSES)), "frozen": True}
]

results = []
cms = {}
for cfg in configs:
    cfg_name = cfg["name"]
    print(f"\n--- Đang huấn luyện cấu hình: {cfg_name} ---")
    tr_loader, va_loader = create_dataloaders(
        data_dir=DATA_DIR,
        train_idx=train_idx,
        val_idx=val_idx,
        augment=cfg["augment"],
        pretrained=cfg["pretrained"],
        batch_size=32,
        seed=42
    )
    seed_everything(42)
    model = cfg["model_fn"]()
    trial_data = train_single_configuration(
        config_name=cfg_name,
        model=model,
        train_loader=tr_loader,
        val_loader=va_loader,
        comp_device=comp_device,
        frozen_backbone=cfg["frozen"],
        epochs=4,
        lr=DEFAULT_LR
    )
    results.append(trial_data)
    from sklearn.metrics import confusion_matrix
    cms[cfg_name] = confusion_matrix(trial_data["y_true"], trial_data["y_pred"], labels=list(range(len(CLASSES))))



## Bước 5: Bảng so sánh tổng hợp 3 cấu hình Thí nghiệm


In [ ]:
df_summary = pd.DataFrame([
    {
        "Cấu hình": res["name"],
        "Số tham số train": f"{res['param_count']:,}",
        "Train Loss": f"{res['history']['train_loss'][-1]:.4f}",
        "Val Loss": f"{res['final_val_loss']:.4f}",
        "Val Accuracy": f"{res['final_val_acc'] * 100:.2f}%",
        "Macro-F1": f"{res['final_val_f1']:.4f}",
        "Giây/epoch": f"{res['avg_sec']:.2f}s"
    }
    for res in results
])
display(df_summary)



## Bước 6: Trực quan hóa đối chiếu Đường cong học tập (3 Subplots)
Lưu biểu đồ vào `docs/day05/task02/image_learning_curves.png` và `bin/day05/task02/results/learning_curves.png`.



In [ ]:
curves_paths = [
    os.path.join(PROJECT_ROOT, "docs", "day05", "task02", "image_learning_curves.png"),
    os.path.join(PROJECT_ROOT, "bin", "day05", "task02", "results", "learning_curves.png")
]
render_comparative_curves(results, curves_paths, epochs=4)
display(Image(curves_paths[0]))



## Bước 7: Trực quan hóa Ma trận nhầm lẫn (Confusion Matrices 3 Subplots)
Lưu biểu đồ vào `docs/day05/task02/image_confusion_matrix.png` và `bin/day05/task02/results/confusion_matrix.png`.



In [ ]:
cm_paths = [
    os.path.join(PROJECT_ROOT, "docs", "day05", "task02", "image_confusion_matrix.png"),
    os.path.join(PROJECT_ROOT, "bin", "day05", "task02", "results", "confusion_matrix.png")
]
render_confusion_matrices(cms, classes=CLASSES, output_paths=cm_paths)
display(Image(cm_paths[0]))



## Bước 8: Trực quan hóa 5 mẫu ảnh dự đoán Đúng và 5 mẫu ảnh dự đoán Sai
Lưu ảnh minh chứng cho từng mô hình vào `docs/day05/task02/` và `bin/day05/task02/results/`.



In [ ]:
for res in results:
    cfg_name = res["name"]
    sample_paths = [
        os.path.join(PROJECT_ROOT, "docs", "day05", "task02", f"image_examples_{cfg_name}.png"),
        os.path.join(PROJECT_ROOT, "bin", "day05", "task02", f"examples_{cfg_name}.png")
    ]
    render_example_predictions(
        raw_dataset=raw_dataset,
        val_idx=val_idx,
        y_true=res["y_true"],
        y_pred=res["y_pred"],
        config_name=cfg_name,
        output_paths=sample_paths,
        classes=CLASSES
    )

display(Image(os.path.join(PROJECT_ROOT, "docs", "day05", "task02", "image_examples_mobilenetv3_small_pretrained.png")))



## Bước 9: Nhận xét chuyên sâu và Bài học kinh nghiệm thực tế

1. **Hiệu ứng của Data Augmentation trên mạng tự học:**
   - Trong 4 epoch đầu, `cnn_augmentation` đạt 39.0% Accuracy so với 42.0% của baseline. Sự tăng độ khó của tập train khiến train loss cao hơn (1.6170 vs 1.4880), nhưng khoảng cách giữa train và val được thu hẹp, giúp điều chuẩn chống overfit hiệu quả.
   - Khi tăng số epoch (ví dụ 15 đến 30 epochs), augmentation sẽ thể hiện ưu thế vượt trội khi baseline bắt đầu phân kỳ quá khớp.

2. **Bước nhảy vọt từ Transfer Learning:**
   - `mobilenetv3_small_pretrained` đạt **83.40% Accuracy** và **0.8350 Macro-F1**, bỏ xa CNN tự học hơn 41 đến 44 điểm phần trăm.
   - Nhờ thừa hưởng đặc trưng tổng quát từ ImageNet, mô hình đạt 78.4% ngay từ epoch 1 và duy trì khả năng tổng quát hóa tuyệt vời.

3. **Đánh đổi về chi phí tính toán:**
   - MobileNet chỉ huấn luyện 10,250 tham số (ít hơn 26 lần), nhưng thời gian mỗi epoch chậm hơn 4 đến 5 lần (3.68s so với 0.72s đến 0.95s) do kích thước ảnh đầu vào $224 \times 224$ lớn gấp 49 lần so với ảnh $32 \times 32$.

